# Sketch vs Visual Face Similarity
**Goal:** Extract features from sketches and visual images, then compute cosine similarity.
- Same person (sketch ↔ own visual) → similarity should be **HIGH** (distance LOW)
- Different person (sketch A ↔ visual B) → similarity should be **LOW** (distance HIGH)

### Expected folder structure:
```
dataset/
├── person_001/
│   ├── visual/   ← multiple photos
│   └── sketch/   ← one sketch
├── person_002/
│   ├── visual/
│   └── sketch/
...
```

In [1]:
# Cell 1 — Imports
import cv2
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
import tensorflow as tf
from tensorflow.keras.models import Model
from keras_vggface.vggface import VGGFace
from keras_vggface.utils import preprocess_input
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity

print('✅ All imports successful')
print('TensorFlow:', tf.__version__)
print('NumPy     :', np.__version__)

/Users/pankaj/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


✅ All imports successful
TensorFlow: 2.13.0
NumPy     : 1.24.3


In [2]:
# Cell 2 — Settings  ← only edit this cell

DATASET_DIR  = '/Users/pankaj/Desktop/sketch_project/LFW_Cropped'    # root folder containing one subfolder per person
VISUAL_DIR   = 'visual'       # name of visual subfolder inside each person folder
SKETCH_DIR   = 'sketch'       # name of sketch subfolder inside each person folder
LAYER        = 'fc6'          # fc6 or fc7
OUTPUT_DIR   = './tantring_features'

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Dataset : {DATASET_DIR}')
print(f'Layer   : {LAYER}')

Dataset : /Users/pankaj/Desktop/sketch_project/LFW_Cropped
Layer   : fc6


In [3]:
# Cell 3 — Load VGGFace model
print('Loading VGGFace...')
base_model = VGGFace(model='vgg16', include_top=True, input_shape=(224, 224, 3))
extractor  = Model(inputs=base_model.input,
                   outputs=base_model.get_layer(LAYER).output)
extractor.trainable = False
print(f'✅ VGGFace loaded  |  output: {extractor.output_shape}')

Loading VGGFace...
✅ VGGFace loaded  |  output: (None, 4096)


In [4]:
# Cell 4 — Preprocessing helpers
EXTS = ('.jpg', '.jpeg', '.png', '.bmp')

def tan_triggs(img_gray, gamma=0.2, sigma0=1, sigma1=2, tau=10.0, alpha=0.1):
    """Tan & Triggs illumination normalisation.
    Paper: Tan & Triggs (2010), IEEE Trans. Image Processing 19(6).
    Eliminates most illumination effects while preserving appearance details."""
    img = img_gray.astype(np.float32)

    # Step 1 — Gamma correction (compresses highlights)
    img = np.power(img + 1e-6, gamma)

    # Step 2 — Difference of Gaussians (band-pass filter removes low-freq lighting)
    g0 = cv2.GaussianBlur(img, (0, 0), sigma0)
    g1 = cv2.GaussianBlur(img, (0, 0), sigma1)
    img = g0 - g1

    # Step 3 — Contrast equalisation, pass 1
    img = img / (np.mean(np.abs(img) ** alpha) ** (1.0 / alpha) + 1e-6)

    # Step 3 — Contrast equalisation, pass 2  (clips extreme values)
    img = img / (np.mean(np.minimum(tau, np.abs(img)) ** alpha) ** (1.0 / alpha) + 1e-6)

    # Step 4 — Tanh normalisation (soft-clips outliers)
    img = tau * np.tanh(img / tau)

    # Rescale to 0–255 uint8 for VGGFace preprocessing
    img = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX)
    return img.astype(np.uint8)


def load_image(path, is_sketch=False):
    """Load, resize, and apply Tan-Triggs uniformly to ALL images
    (both sketches and visuals) — as advised by professor:
    apply the same preprocessing to the entire dataset so that
    both modalities live in the same feature space."""
    img = cv2.imread(path)
    if img is None:
        raise FileNotFoundError(f'Cannot read: {path}')
    if len(img.shape) == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    elif img.shape[2] == 1:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)

    img = cv2.resize(img, (224, 224))

    # Convert to grayscale, apply Tan-Triggs, then back to BGR
    # Applied identically to BOTH sketches and photos
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    tt   = tan_triggs(gray)
    img  = cv2.cvtColor(tt, cv2.COLOR_GRAY2BGR)

    return img.astype(np.float32)


def extract_feature(img_array):
    x = np.expand_dims(img_array, axis=0)
    x = preprocess_input(x, version=1)
    feat = extractor.predict(x, verbose=0).flatten()
    return normalize(feat.reshape(1, -1))[0]

print('✅ Helpers defined (Tan-Triggs preprocessing enabled for ALL images)')


✅ Helpers defined (Tan-Triggs preprocessing enabled for ALL images)


In [5]:
# Cell 5 — Scan dataset and collect all persons
persons = sorted([
    p for p in os.listdir(DATASET_DIR)
    if os.path.isdir(os.path.join(DATASET_DIR, p))
])

print(f'Found {len(persons)} persons:')
for p in persons[:5]:
    print(f'  {p}')
if len(persons) > 5:
    print(f'  ... and {len(persons)-5} more')

Found 2753 persons:
  AJ_Cook
  AJ_Lamas
  Aaron_Eckhart
  Aaron_Guiel
  Aaron_Patterson
  ... and 2748 more


In [ ]:
# Cell 6 — Extract features for ALL persons
# For each person:
#   - sketch_feat       : single feature vector from the sketch
#   - visual_feat_mean  : mean of all visual feature vectors (one per photo)

sketch_features = {}   # person_id → 4096-dim vector
visual_features = {}   # person_id → 4096-dim vector (mean of all visuals)
failed          = []

for person in tqdm(persons, desc='Extracting features'):
    person_path  = os.path.join(DATASET_DIR, person)
    sketch_path  = os.path.join(person_path, SKETCH_DIR)
    visual_path  = os.path.join(person_path, VISUAL_DIR)

    try:
        # --- Sketch (one image) ---
        sketch_files = [f for f in os.listdir(sketch_path) if f.lower().endswith(EXTS)]
        if not sketch_files:
            raise FileNotFoundError(f'No sketch found for {person}')
        sketch_img  = load_image(os.path.join(sketch_path, sketch_files[0]), is_sketch=True)
        sketch_feat = extract_feature(sketch_img)

        # --- Visuals (multiple images → mean feature) ---
        visual_files = [f for f in os.listdir(visual_path) if f.lower().endswith(EXTS)]
        if not visual_files:
            raise FileNotFoundError(f'No visuals found for {person}')

        v_feats = []
        for vf in visual_files:
            vimg  = load_image(os.path.join(visual_path, vf), is_sketch=False)
            vfeat = extract_feature(vimg)
            v_feats.append(vfeat)

        # Mean of all visual features → represents the person
        visual_feat = normalize(np.mean(v_feats, axis=0).reshape(1,-1))[0]

        sketch_features[person] = sketch_feat
        visual_features[person] = visual_feat

    except Exception as e:
        failed.append((person, str(e)))

print(f'\n✅ Extracted : {len(sketch_features)} persons')
print(f'❌ Failed    : {len(failed)}')
for f in failed:
    print(f'   {f[0]} → {f[1]}')

Extracting features:   0%|          | 0/2753 [00:00<?, ?it/s]

In [ ]:
# Cell 7 — Build similarity matrix
# Rows    = sketches  (person A sketch, person B sketch ...)
# Columns = visuals   (person A visual, person B visual ...)
# Value   = cosine similarity between sketch i and visual j
#
# IDEAL result:
#   Diagonal (same person) → HIGH similarity ~1.0
#   Off-diagonal           → LOW  similarity ~0.0

person_ids     = list(sketch_features.keys())
sketch_matrix  = np.stack([sketch_features[p] for p in person_ids])  # (N, 4096)
visual_matrix  = np.stack([visual_features[p] for p in person_ids])  # (N, 4096)

# Full N×N similarity matrix
sim_matrix = cosine_similarity(sketch_matrix, visual_matrix)  # (N, N)

print(f'Similarity matrix shape: {sim_matrix.shape}')
print(f'\nDiagonal (same person) stats:')
diag = np.diag(sim_matrix)
print(f'  Mean : {diag.mean():.4f}')
print(f'  Min  : {diag.min():.4f}')
print(f'  Max  : {diag.max():.4f}')

# Off-diagonal
mask     = ~np.eye(len(person_ids), dtype=bool)
off_diag = sim_matrix[mask]
print(f'\nOff-diagonal (different person) stats:')
print(f'  Mean : {off_diag.mean():.4f}')
print(f'  Min  : {off_diag.min():.4f}')
print(f'  Max  : {off_diag.max():.4f}')

In [ ]:
# Cell 8 — Plot similarity matrix (fixed size)
N = len(person_ids)

# Cap figure size to maximum 20x20 inches regardless of N
fig_size = min(20, max(8, N // 5))

plt.figure(figsize=(fig_size, fig_size))
sns.heatmap(
    sim_matrix,
    xticklabels=False,      # hide labels if too many persons
    yticklabels=False,
    cmap='RdYlGn',
    vmin=0, vmax=1,
    annot=False,            # never annotate — too many cells
    linewidths=0,           # no grid lines — too slow for large N
    square=True
)
plt.title(f'Sketch vs Visual Cosine Similarity ({LAYER.upper()}) — {N} persons', fontsize=13)
plt.xlabel('Visual Identity')
plt.ylabel('Sketch Identity')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, f'similarity_matrix_{LAYER}.png'), dpi=100, bbox_inches='tight')
plt.show()
print(f'✅ Saved to {OUTPUT_DIR}/similarity_matrix_{LAYER}.png')

In [ ]:
# Cell 9 — Diagonal vs Off-diagonal distribution plot
# Good model: two peaks clearly separated

diag     = np.diag(sim_matrix)
off_diag = sim_matrix[~np.eye(N, dtype=bool)]

plt.figure(figsize=(10, 4))
plt.hist(off_diag, bins=50, alpha=0.7, color='tomato',    label='Different person (should be LOW)')
plt.hist(diag,     bins=20, alpha=0.9, color='steelblue', label='Same person (should be HIGH)')
plt.axvline(diag.mean(),     color='steelblue', linestyle='--', linewidth=1.5, label=f'Same person mean: {diag.mean():.3f}')
plt.axvline(off_diag.mean(), color='tomato',    linestyle='--', linewidth=1.5, label=f'Diff person mean: {off_diag.mean():.3f}')
plt.xlabel('Cosine Similarity')
plt.ylabel('Count')
plt.title(f'Similarity Distribution — {LAYER.upper()}')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, f'similarity_distribution_{LAYER}.png'), dpi=150)
plt.show()

In [ ]:
# Cell 10 — Rank-1 retrieval accuracy
# For each sketch, find the most similar visual
# If it matches the correct person → correct retrieval

correct = 0
for i, person in enumerate(person_ids):
    best_match_idx = np.argmax(sim_matrix[i])   # most similar visual
    if person_ids[best_match_idx] == person:
        correct += 1

accuracy = correct / len(person_ids) * 100
print(f'Rank-1 Retrieval Accuracy ({LAYER.upper()}): {correct}/{len(person_ids)} = {accuracy:.1f}%')
print()
print('What this means:')
print('  100% → every sketch correctly matched to its own visual')
print('  50%  → half correct (random chance for 2 people = 50%)')
print('  For N persons, random chance =', round(100/len(person_ids), 1), '%')

In [ ]:
# Cell 11 — Save everything
np.save(os.path.join(OUTPUT_DIR, f'sim_matrix_{LAYER}.npy'),  sim_matrix)
np.save(os.path.join(OUTPUT_DIR, 'person_ids.npy'), np.array(person_ids))

print(f'✅ Saved sim_matrix_{LAYER}.npy  →  shape {sim_matrix.shape}')
print(f'✅ Saved person_ids.npy          →  {len(person_ids)} persons')

In [ ]:
# Zoom into first 50 persons only — so diagonal is visible
N_SHOW = 50

plt.figure(figsize=(12, 10))
sns.heatmap(
    sim_matrix[:N_SHOW, :N_SHOW],
    xticklabels=person_ids[:N_SHOW],
    yticklabels=person_ids[:N_SHOW],
    cmap='RdYlGn',
    vmin=0, vmax=1,
    annot=True,
    fmt='.2f',
    linewidths=0.5,
    square=True,
    annot_kws={"size": 6}
)
plt.title(f'Similarity Matrix — First {N_SHOW} persons (FC7)', fontsize=13)
plt.xlabel('Visual Identity')
plt.ylabel('Sketch Identity')
plt.xticks(rotation=90, fontsize=6)
plt.yticks(rotation=0, fontsize=6)
plt.tight_layout()
plt.show()

In [ ]:
diag     = np.diag(sim_matrix)
mask     = ~np.eye(len(person_ids), dtype=bool)
off_diag = sim_matrix[mask]

print(f'Same person (diagonal)     mean: {diag.mean():.4f}')
print(f'Different person (off-diag) mean: {off_diag.mean():.4f}')
print(f'Separation gap: {diag.mean() - off_diag.mean():.4f}')
print()
print('Good result  → gap > 0.3')
print('Okay result  → gap 0.1–0.3')  
print('Weak result  → gap < 0.1')

---
## How to read the similarity matrix
```
         Visual_A  Visual_B  Visual_C
Sketch_A  [ 0.92     0.21      0.18 ]   ← A matched correctly ✅
Sketch_B  [ 0.19     0.88      0.22 ]   ← B matched correctly ✅
Sketch_C  [ 0.20     0.31      0.85 ]   ← C matched correctly ✅
```
- **Bright diagonal** = model working well
- **Dark off-diagonal** = model correctly separating identities
- **Rank-1 accuracy** = % of sketches that found correct visual as top match

## Change layer
Change `LAYER = 'fc6'` in Cell 2 and re-run to compare FC6 vs FC7.